# Galgame MTL — GGUF 模型與執行檔部署工具（免費 CPU 執行階段即可）

**用途：** 將鎖定版本的 GGUF 權重檔及預編譯 L4 `llama-server`（CUDA `b12019`）自動下載並以雙重 SHA-256 校驗部署到尚未跑過本專案的 Google Drive。

**使用方式：**
1. 確認右上角執行階段為 **CPU**（完全不花 GPU 算力單元）。
2. 勾選要部署的模型（本分支預設 `DEPLOY_SAKURA = True`，亦可順便勾選 `DEPLOY_HYMT2` 一次鋪好兩個；預設 `DEPLOY_LLAMA_BIN = True` 會在免費 CPU 先快取好 L4 預編譯檔，L4 開機免編譯）。
3. 按 **`Ctrl+F9`（全部執行）**，跳出 Google Drive 授權點「允許」後即可放著跑。

> 已存在於 Drive 且「位元組大小 + SHA-256」完全吻合的檔案會自動略過；不吻合或殘留檔會刪除重拉（最多 3 次，含 `hf-mirror.com` 備援）。

In [ ]:
#@title 1. 部署設定 { display-mode: "form" }
DEPLOY_SAKURA = True #@param {type:"boolean"}
DEPLOY_HYMT2 = False #@param {type:"boolean"}
DEPLOY_LLAMA_BIN = True #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/galgame_mtl" #@param {type:"string"}
AUTO_DISCONNECT = False #@param {type:"boolean"}

In [ ]:
#@title 2. 掛載雲端硬碟 -> 建目錄 -> 下載與雙重 SHA-256 校驗 -> Flush
import hashlib
import os
import shutil
import subprocess
import tarfile
import time
from pathlib import Path, PurePosixPath
from google.colab import drive, runtime

MODELS = {
    "sakura": {
        "filename": "Sakura-Galtransl-14B-v3.8.gguf",
        "size_bytes": 12124681536,
        "sha256": "6e32fcf5bfdf78114745d2bf6bd87fb107ad7f68a611f4c2a886a4378846e334",
        "urls": [
            "https://huggingface.co/SakuraLLM/Sakura-GalTransl-14B-v3.8/resolve/10bab61b45d8970dc5c5fd7a4aec920d49a3cd8e/Sakura-Galtransl-14B-v3.8.gguf",
            "https://hf-mirror.com/SakuraLLM/Sakura-GalTransl-14B-v3.8/resolve/10bab61b45d8970dc5c5fd7a4aec920d49a3cd8e/Sakura-Galtransl-14B-v3.8.gguf",
        ],
    },
    "hymt2": {
        "filename": "Hy-MT2-30B-A3B-uncensored-heretic.i1-Q4_K_M.gguf",
        "size_bytes": 18236703488,
        "sha256": "b0358d0319aae3f821fef4989bd85f409336139bfc7db59fd049848b9882321c",
        "urls": [
            "https://huggingface.co/mradermacher/Hy-MT2-30B-A3B-uncensored-heretic-i1-GGUF/resolve/862b7d66847b339c0594b8de39d38b035ee75f25/Hy-MT2-30B-A3B-uncensored-heretic.i1-Q4_K_M.gguf",
            "https://hf-mirror.com/mradermacher/Hy-MT2-30B-A3B-uncensored-heretic-i1-GGUF/resolve/862b7d66847b339c0594b8de39d38b035ee75f25/Hy-MT2-30B-A3B-uncensored-heretic.i1-Q4_K_M.gguf",
        ],
    },
}

PREBUILT_BIN_ASSET = "llama-server-l4-b12019.tar.gz"
PREBUILT_BIN_URL = (
    "https://github.com/poynt2005/hanhua-mtl/releases/download/"
    "colab-l4-llama-b12019/llama-server-l4-b12019.tar.gz"
)
PREBUILT_BIN_SHA_URL = PREBUILT_BIN_URL + ".sha256"

UPSTREAM_PREBUILT_SPECS = [
    {
        "name": "llama-b12019-bin-ubuntu-cuda-12.8-x64.tar.gz",
        "size_bytes": 182780961,
        "sha256": "0295172839d1ff69c4e83d69b5cd3cad81442c463004495710de694968b7fab8",
        "url": "https://github.com/ggml-org/llama.cpp/releases/download/b12019/llama-b12019-bin-ubuntu-cuda-12.8-x64.tar.gz",
    },
    {
        "name": "cudart-llama-b12019-bin-ubuntu-cuda-12.8-x64.tar.gz",
        "size_bytes": 594370124,
        "sha256": "842484ddbe6a6b29d9587b2295d0310ecf4e9825f28f129e019c4624168ec82e",
        "url": "https://github.com/ggml-org/llama.cpp/releases/download/b12019/cudart-llama-b12019-bin-ubuntu-cuda-12.8-x64.tar.gz",
    },
]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            chunk = fh.read(8 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def verify_file(path: Path, exp_size: int, exp_sha: str, label: str) -> bool:
    if not path.is_file():
        return False
    sz = path.stat().st_size
    if sz != exp_size:
        print(f"[{label}] 大小不符: {sz} != {exp_size}")
        return False
    print(f"[{label}] 計算 SHA-256 ({sz / 1e9:.2f} GB)...")
    got = sha256_file(path)
    if got != exp_sha:
        print(f"[{label}] SHA-256 不符: {got} != {exp_sha}")
        return False
    print(f"[{label}] SHA-256 驗證通過: {got[:16]}...")
    return True

def download_to_local(urls, dst: Path, exp_size: int) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    for u in urls:
        if dst.exists():
            dst.unlink()
        print(f"[download] 嘗試拉取: {u}")
        rc = subprocess.run(
            ["curl", "-fL", "--retry", "3", "--retry-delay", "5", "-o", str(dst), u]
        ).returncode
        if rc == 0 and dst.is_file() and dst.stat().st_size == exp_size:
            return
        print(f"[download] 失敗或大小不符 (rc={rc})，嘗試下一個來源...")
    raise RuntimeError(f"所有來源皆下載失敗: {dst.name}")

def deploy_one(key: str, models_dir: Path) -> None:
    spec = MODELS[key]
    dst = models_dir / spec["filename"]
    part = models_dir / f".{spec['filename']}.part"
    tmp_local = Path("/content/tmp") / spec["filename"]

    if verify_file(dst, spec["size_bytes"], spec["sha256"], f"{key}/drive"):
        print(f"[{key}] 已存在且通過 SHA-256，略過下載。")
        return
    if dst.exists():
        print(f"[{key}] 移除 Drive 毀損檔: {dst.name}")
        dst.unlink()
    if part.exists():
        part.unlink()

    for attempt in range(1, 4):
        print(f"\n[{key}] === 第 {attempt}/3 次部署嘗試 ===")
        try:
            download_to_local(spec["urls"], tmp_local, spec["size_bytes"])
            if not verify_file(tmp_local, spec["size_bytes"], spec["sha256"], f"{key}/local"):
                raise RuntimeError("本地 SSD SHA-256 校驗失敗")
            print(f"[{key}] 複製至 Drive 暫存檔: {part.name}...")
            shutil.copyfile(tmp_local, part)
            tmp_local.unlink(missing_ok=True)
            if not verify_file(part, spec["size_bytes"], spec["sha256"], f"{key}/drive-part"):
                raise RuntimeError("Drive 端 SHA-256 複驗失敗")
            os.replace(part, dst)
            print(f"[{key}] 部署完成 -> {dst}")
            return
        except Exception as exc:
            print(f"[{key}] 第 {attempt} 次失敗: {exc}")
            tmp_local.unlink(missing_ok=True)
            part.unlink(missing_ok=True)
            time.sleep(3)
    raise RuntimeError(f"[{key}] 連續 3 次部署均失敗，請檢查網路或 Drive 剩餘空間。")

def deploy_llama_bin(bin_dir: Path) -> None:
    dst = bin_dir / PREBUILT_BIN_ASSET
    dst_sha = bin_dir / f"{PREBUILT_BIN_ASSET}.sha256"
    part = bin_dir / f".{PREBUILT_BIN_ASSET}.part"
    tmp_dir = Path("/content/tmp/llama_bin")

    def _parse_sha(text: str):
        tok = text.strip().split()[0].lower() if text.strip() else ""
        return tok if len(tok) == 64 and all(c in "0123456789abcdef" for c in tok) else None

    if dst.is_file() and dst_sha.is_file():
        exp_local = _parse_sha(dst_sha.read_text(encoding="utf-8-sig", errors="replace"))
        if exp_local and sha256_file(dst) == exp_local:
            print(f"[bin] 已存在且通過 SHA-256，略過下載: {dst.name}")
            return

    shutil.rmtree(tmp_dir, ignore_errors=True)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    part.unlink(missing_ok=True)
    try:
        local_tar = tmp_dir / PREBUILT_BIN_ASSET
        sha_res = subprocess.run(["curl", "-fsSL", PREBUILT_BIN_SHA_URL], capture_output=True, text=True)
        exp_sha = _parse_sha(sha_res.stdout) if sha_res.returncode == 0 else None
        if exp_sha:
            if dst.is_file() and sha256_file(dst) == exp_sha:
                dst_sha.write_text(f"{exp_sha}  {PREBUILT_BIN_ASSET}\n", encoding="utf-8")
                print(f"[bin] 已存在且符合 Release SHA-256: {dst.name}")
                return
            print(f"[bin] 從 GitHub Release 拉取預編譯包: {PREBUILT_BIN_ASSET}...")
            rc = subprocess.run(["curl", "-fL", "--retry", "3", "--retry-delay", "3", "-o", str(local_tar), PREBUILT_BIN_URL]).returncode
            if rc == 0 and local_tar.is_file() and sha256_file(local_tar) == exp_sha:
                shutil.copyfile(local_tar, part)
                if sha256_file(part) == exp_sha:
                    os.replace(part, dst)
                    dst_sha.write_text(f"{exp_sha}  {PREBUILT_BIN_ASSET}\n", encoding="utf-8")
                    print(f"[bin] 預編譯 L4 llama-server 部署完成 -> {dst}")
                    return
            print("[bin] Release 預編譯包校驗失敗，改用官方 b12019 CUDA 12.8 預編譯包...")
        else:
            print("[bin] Release 尚未建立，改從官方 b12019 CUDA 12.8 打包快取...")

        ext_dir = tmp_dir / "extracted"
        pack_dir = tmp_dir / "pack"
        ext_dir.mkdir(parents=True, exist_ok=True)
        pack_dir.mkdir(parents=True, exist_ok=True)
        for idx, spec in enumerate(UPSTREAM_PREBUILT_SPECS):
            arc = tmp_dir / spec["name"]
            download_to_local([spec["url"]], arc, spec["size_bytes"])
            if not verify_file(arc, spec["size_bytes"], spec["sha256"], spec["name"]):
                raise RuntimeError(f"SHA-256 不符: {spec['name']}")
            with tarfile.open(arc, "r:gz") as tf:
                members = [m for m in tf.getmembers() if not m.name.startswith("/") and ".." not in PurePosixPath(m.name).parts]
                tf.extractall(ext_dir / f"arc_{idx}", members=members)
            arc.unlink(missing_ok=True)
        exe_src = next((p for p in ext_dir.rglob("llama-server") if p.is_file()), None)
        if exe_src is None:
            raise RuntimeError("官方壓縮包內找不到 llama-server")
        shutil.copyfile(exe_src, pack_dir / "llama-server")
        (pack_dir / "llama-server").chmod(0o755)
        seen = {"llama-server"}
        for so_p in sorted(ext_dir.rglob("*.so*")):
            if (so_p.is_file() or so_p.is_symlink()) and so_p.name not in seen:
                dst_so = pack_dir / so_p.name
                if so_p.is_symlink():
                    tgt = os.readlink(so_p)
                    if os.path.basename(tgt) == tgt:
                        os.symlink(tgt, dst_so)
                    elif so_p.resolve().is_file():
                        shutil.copyfile(so_p.resolve(), dst_so)
                else:
                    shutil.copyfile(so_p, dst_so)
                seen.add(so_p.name)
        with tarfile.open(local_tar, "w:gz") as tf:
            for item in pack_dir.iterdir():
                tf.add(item, arcname=item.name)
        pack_sha = sha256_file(local_tar)
        shutil.copyfile(local_tar, part)
        if sha256_file(part) != pack_sha:
            raise RuntimeError("Drive 端 bin 暫存檔 SHA-256 複驗失敗")
        os.replace(part, dst)
        dst_sha.write_text(f"{pack_sha}  {PREBUILT_BIN_ASSET}\n", encoding="utf-8")
        print(f"[bin] 官方 CUDA 12.8 預編譯包已快取 -> {dst} (L4 開機免編譯)")
    except Exception as exc:
        print(f"[bin] warn: 預先快取 llama-server 失敗 ({exc})；L4 執行時會再嘗試自動拉取。")
    finally:
        part.unlink(missing_ok=True)
        shutil.rmtree(tmp_dir, ignore_errors=True)

mounted = False
try:
    drive.mount("/content/drive", force_remount=False)
    mounted = True
    root = Path(DRIVE_ROOT)
    for rel in (
        "bin",
        "models",
        "scripts/sakura",
        "scripts/hymt2",
        "sakura/input",
        "sakura/work",
        "sakura/output",
        "hymt2/input",
        "hymt2/work",
        "hymt2/output",
    ):
        (root / rel).mkdir(parents=True, exist_ok=True)
    print(f"[init] 目錄結構就緒: {root}")

    selected = [k for k, on in (("sakura", DEPLOY_SAKURA), ("hymt2", DEPLOY_HYMT2)) if on]
    if not selected:
        print("[warn] 未勾選任何模型，僅完成目錄建立。")
    for k in selected:
        deploy_one(k, root / "models")
    if DEPLOY_LLAMA_BIN:
        deploy_llama_bin(root / "bin")
    print("\n[done] 全部選取項目均已就緒並通過雙重 SHA-256 驗證！")
finally:
    shutil.rmtree("/content/tmp", ignore_errors=True)
    if mounted:
        try:
            print("[flush] 正在將 Google Drive 緩衝同步回雲端 (請勿關閉)...")
            drive.flush_and_unmount()
            print("[flush] 同步完成！")
        except Exception as exc:
            print(f"[flush] warn: {exc}")
    if AUTO_DISCONNECT:
        runtime.unassign()
